<a href="https://colab.research.google.com/github/yogeshwardev/CSA6102-Digital_forensics-/blob/main/lab%20programs%20outputs%20from%20colab%2017%20to%2028.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import hashlib

evidence_dir = "digital_evidence"
os.makedirs(evidence_dir, exist_ok=True)
sample_evidence = {
    "email_log.txt": "From: attacker@mail.com\nTo: victim@mail.com\nSubject: Invoice\nPlease pay immediately.",
    "browser_history.txt": "http://malicious-site.com/login\nhttp://bank.com/transfer",
    "chat_message.txt": "Hey, did you send the file?",
    "system_log.txt": "2026-07-24 10:15:32 - USB device connected: E:\\",
}

for filename, content in sample_evidence.items():
    with open(os.path.join(evidence_dir, filename), "w") as f:
        f.write(content)

def catalog_evidence(folder):
    catalog = []
    for fname in sorted(os.listdir(folder)):
        path = os.path.join(folder, fname)
        stat = os.stat(path)
        with open(path, "rb") as f:
            data = f.read()
        catalog.append({
            "filename": fname,
            "size_bytes": stat.st_size,
            "md5": hashlib.md5(data).hexdigest(),
        })
    return catalog

evidence_catalog = catalog_evidence(evidence_dir)
for item in evidence_catalog:
    print(item)

def test_experiment17():
    assert os.path.isdir(evidence_dir), "Evidence directory should exist"
    assert len(evidence_catalog) == len(sample_evidence), "All evidence files should be catalogued"
    filenames = [e["filename"] for e in evidence_catalog]
    for expected in sample_evidence:
        assert expected in filenames, f"{expected} missing from catalog"
    for entry in evidence_catalog:
        assert entry["size_bytes"] > 0
        assert len(entry["md5"]) == 32
    print("Experiment 17: All test cases passed.")

test_experiment17()


{'filename': 'browser_history.txt', 'size_bytes': 56, 'md5': '7804eb386588e70e52f8e44985aae9ab'}
{'filename': 'chat_message.txt', 'size_bytes': 27, 'md5': 'e5f0758dbf1fe3c48005ea2560fa5279'}
{'filename': 'email_log.txt', 'size_bytes': 84, 'md5': '94d472f8d614b91afc7ab2f56ddb435d'}
{'filename': 'system_log.txt', 'size_bytes': 47, 'md5': '5731691a0851b79317771c6e09f5c53b'}
Experiment 17: All test cases passed.


In [2]:
import os
import psutil

evidence_dir = "digital_evidence"
os.makedirs(evidence_dir, exist_ok=True)
sample_evidence = {
    "email_log.txt": "From: attacker@mail.com\nTo: victim@mail.com\nSubject: Invoice\nPlease pay immediately.",
    "browser_history.txt": "http://malicious-site.com/login\nhttp://bank.com/transfer",
    "chat_message.txt": "Hey, did you send the file?",
    "system_log.txt": "2026-07-24 10:15:32 - USB device connected: E:\\",
}

for filename, content in sample_evidence.items():
    with open(os.path.join(evidence_dir, filename), "w") as f:
        f.write(content)

def capture_volatile_evidence():
    processes = [p.info for p in psutil.process_iter(['pid', 'name'])]
    return {
        "running_process_count": len(processes),
        "sample_processes": processes[:5],
    }

def capture_nonvolatile_evidence(folder):
    files = sorted(os.listdir(folder))
    return {"disk_files": files, "file_count": len(files)}

volatile_snapshot = capture_volatile_evidence()
nonvolatile_snapshot = capture_nonvolatile_evidence(evidence_dir)

print("Volatile evidence -> running processes:", volatile_snapshot["running_process_count"])
print("Non-volatile evidence -> disk files:", nonvolatile_snapshot)

def test_experiment18():
    assert volatile_snapshot["running_process_count"] > 0, "There should be at least one running process"
    assert isinstance(volatile_snapshot["sample_processes"], list)
    assert nonvolatile_snapshot["file_count"] == len(sample_evidence)
    snapshot2 = capture_nonvolatile_evidence(evidence_dir)
    assert snapshot2 == nonvolatile_snapshot, "Non-volatile evidence must remain unchanged between reads"
    print("Experiment 18: All test cases passed.")

test_experiment18()


Volatile evidence -> running processes: 13
Non-volatile evidence -> disk files: {'disk_files': ['browser_history.txt', 'chat_message.txt', 'email_log.txt', 'system_log.txt'], 'file_count': 4}
Experiment 18: All test cases passed.


In [3]:
import os

os.makedirs("acquisition_demo/source", exist_ok=True)
source = "acquisition_demo/source"

with open(os.path.join(source, "report.docx"), "w") as f:
    f.write("Confidential quarterly report.")
with open(os.path.join(source, "photo.jpg"), "w") as f:
    f.write("FAKEJPEGDATA")

raw_disk = b"REPORTDOCX_CONTENT" + b"\x00" * 20 + b"DELETED_INVOICE_DATA" + b"\x00" * 10

def physical_acquisition(raw_bytes):
    return bytes(raw_bytes) # every bit: used + unused + deleted

def logical_acquisition(folder):
    return {f: open(os.path.join(folder, f), "rb").read() for f in os.listdir(folder)}

def sparse_acquisition(folder, targets):
    return {f: open(os.path.join(folder, f), "rb").read() for f in targets if f in os.listdir(folder)}

physical_image = physical_acquisition(raw_disk)
logical_image = logical_acquisition(source)
sparse_image = sparse_acquisition(source, ["report.docx"])

print("Physical image size:", len(physical_image), "bytes")
print("Logical image files:", list(logical_image.keys()))
print("Sparse image files:", list(sparse_image.keys()))

def test_experiment19():
    assert physical_image == raw_disk, "Physical acquisition must be an exact byte-for-byte copy"
    assert b"DELETED_INVOICE_DATA" in physical_image, "Physical acquisition must include deleted/unused regions"
    assert set(logical_image.keys()) == {"report.docx", "photo.jpg"}, "Logical acquisition should copy all visible files"
    assert set(sparse_image.keys()) == {"report.docx"}, "Sparse acquisition should copy only the targeted file"
    print("Experiment 19: All test cases passed.")

test_experiment19()


Physical image size: 68 bytes
Logical image files: ['photo.jpg', 'report.docx']
Sparse image files: ['report.docx']
Experiment 19: All test cases passed.


In [4]:
import time
import psutil
import json

def live_acquisition():
    return {
        "timestamp": time.time(),
        "running_processes": len(psutil.pids()),
        "cpu_percent": psutil.cpu_percent(interval=0.1),
    }

def dead_acquisition(snapshot_file):
    with open(snapshot_file) as f:
        return json.load(f)

dead_snapshot_path = "system_snapshot.json"
static_snapshot = {"hard_disk_files": ["a.txt", "b.txt"], "ram_data": None}
with open(dead_snapshot_path, "w") as f:
    json.dump(static_snapshot, f)

live_result = live_acquisition()
dead_result = dead_acquisition(dead_snapshot_path)

print("Live acquisition:", live_result)
print("Dead acquisition:", dead_result)

def test_experiment20():
    assert live_result["running_processes"] > 0, "Live acquisition must capture running processes"
    assert isinstance(live_result["cpu_percent"], float), "Live acquisition should return live CPU usage"
    assert dead_result["ram_data"] is None, "Dead acquisition can never recover RAM data"
    assert dead_result == static_snapshot, "Dead acquisition should exactly match the stored snapshot"
    print("Experiment 20: All test cases passed.")

test_experiment20()


Live acquisition: {'timestamp': 1785227989.1426654, 'running_processes': 13, 'cpu_percent': 10.5}
Dead acquisition: {'hard_disk_files': ['a.txt', 'b.txt'], 'ram_data': None}
Experiment 20: All test cases passed.


In [5]:
raw_disk_full = b"FILE1DATA" + b"\x00" * 15 + b"DELETED_FILE_DATA" + b"\x00" * 15 + b"FREE_SPACE_00000"

def create_forensic_image(raw_bytes):
    return bytes(raw_bytes) # bit-for-bit: files + deleted data + free space

def create_duplication(raw_bytes, active_regions):
    return b"".join(raw_bytes[start:end] for start, end in active_regions)

active_regions = [(0, 9)] # only the live "FILE1DATA" region
forensic_image = create_forensic_image(raw_disk_full)
duplication_copy = create_duplication(raw_disk_full, active_regions)

print("Forensic image size:", len(forensic_image))
print("Duplication size:", len(duplication_copy))

def test_experiment21():
    assert len(forensic_image) == len(raw_disk_full), "Imaging must capture the full storage device size"
    assert b"DELETED_FILE_DATA" in forensic_image, "Imaging must include deleted data"
    assert b"DELETED_FILE_DATA" not in duplication_copy, "Duplication must exclude deleted data"
    assert len(duplication_copy) < len(forensic_image), "Duplication should be smaller than a full image"
    print("Experiment 21: All test cases passed.")

test_experiment21()


Forensic image size: 72
Duplication size: 9
Experiment 21: All test cases passed.


In [6]:
import os
import hashlib

original_path = "original_evidence.bin"
with open(original_path, "wb") as f:
    f.write(os.urandom(1024)) # simulate a small storage device

def bit_stream_copy(src_path, dst_path):
    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        dst.write(src.read())

copy_path = "bitstream_copy.bin"
bit_stream_copy(original_path, copy_path)

def sha256_of_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

original_hash = sha256_of_file(original_path)
copy_hash = sha256_of_file(copy_path)

print("Original hash:", original_hash)
print("Copy hash: ", copy_hash)

def test_experiment22():
    assert os.path.getsize(original_path) == os.path.getsize(copy_path), "Bit stream copy must match original size exactly"
    assert original_hash == copy_hash, "Hashes must match, proving the copy is forensically identical"
    print("Experiment 22: All test cases passed.")

test_experiment22()


Original hash: eebbd9f07dc449c3bb23bfd8ce23495ad3aa47346cd61f07ee9b2c02b1429ce7
Copy hash:  eebbd9f07dc449c3bb23bfd8ce23495ad3aa47346cd61f07ee9b2c02b1429ce7
Experiment 22: All test cases passed.


In [7]:
import hashlib

def compute_hashes(data: bytes):
    return {
        "MD5": hashlib.md5(data).hexdigest(),
        "SHA1": hashlib.sha1(data).hexdigest(),
        "SHA256": hashlib.sha256(data).hexdigest(),
    }

original_data = b"hello"
tampered_data = b"Hello"

hashes_original = compute_hashes(original_data)
hashes_tampered = compute_hashes(tampered_data)
hashes_original_repeat = compute_hashes(original_data)

print("hello ->", hashes_original)
print("Hello ->", hashes_tampered)

def test_experiment23():
    assert hashes_original == hashes_original_repeat, "Same input must always produce the same hash"
    assert hashes_original["MD5"] != hashes_tampered["MD5"], "A single-character change must alter the MD5 hash"
    assert hashes_original["SHA256"] != hashes_tampered["SHA256"], "A single-character change must alter the SHA-256 hash"
    print("Experiment 23: All test cases passed.")

test_experiment23()


hello -> {'MD5': '5d41402abc4b2a76b9719d911017c592', 'SHA1': 'aaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d', 'SHA256': '2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824'}
Hello -> {'MD5': '8b1a9953c4611296a827abf8c47804d7', 'SHA1': 'f7ff9e8b7bb2e09b70935a5d785e0cc5d9d0abf0', 'SHA256': '185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969'}
Experiment 23: All test cases passed.


In [8]:
import os

def make_fake_jpeg(payload: bytes):
    return b"\xff\xd8\xff" + payload + b"\xff\xd9"

jpeg1 = make_fake_jpeg(b"PHOTO_OF_SUSPECT_CAR")
jpeg2 = make_fake_jpeg(b"CCTV_FRAME_CAPTURE")
raw_disk_blob = os.urandom(30) + jpeg1 + os.urandom(40) + jpeg2 + os.urandom(20)

def carve_jpegs(blob: bytes):
    recovered = []
    start_marker, end_marker = b"\xff\xd8\xff", b"\xff\xd9"
    pos = 0
    while True:
        start = blob.find(start_marker, pos)
        if start == -1:
            break
        end = blob.find(end_marker, start)
        if end == -1:
            break
        end += len(end_marker)
        recovered.append(blob[start:end])
        pos = end
    return recovered

recovered_files = carve_jpegs(raw_disk_blob)
print("Recovered", len(recovered_files), "file(s) via carving")

def test_experiment24():
    assert len(recovered_files) == 2, "File carving should recover exactly 2 embedded JPEGs"
    assert recovered_files[0] == jpeg1, "First recovered file must exactly match the original embedded JPEG"
    assert recovered_files[1] == jpeg2, "Second recovered file must exactly match the original embedded JPEG"
    print("Experiment 24: All test cases passed.")

test_experiment24()


Recovered 2 file(s) via carving
Experiment 24: All test cases passed.


In [9]:
class FATFileEntry:
    def __init__(self, name, size):
        self.name = name
        self.size = size
        # FAT intentionally has no permissions or journal attribute

class NTFSFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"

class EXTFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"

fat_file = FATFileEntry("data.txt", 1024)
ntfs_file = NTFSFileEntry("data.txt", 1024)
ext_file = EXTFileEntry("data.txt", 1024)

print("FAT has permissions attribute:", hasattr(fat_file, "permissions"))
print("NTFS has permissions attribute:", hasattr(ntfs_file, "permissions"))
print("EXT has permissions attribute:", hasattr(ext_file, "permissions"))

def test_experiment25():
    assert not hasattr(fat_file, "permissions"), "FAT should not support file permissions"
    assert not hasattr(fat_file, "journal_entry"), "FAT should not support journaling"
    assert hasattr(ntfs_file, "permissions") and hasattr(ntfs_file, "journal_entry"), "NTFS should support permissions and journaling"
    assert hasattr(ext_file, "permissions") and hasattr(ext_file, "journal_entry"), "EXT should support permissions and journaling"
    print("Experiment 25: All test cases passed.")

test_experiment25()


FAT has permissions attribute: False
NTFS has permissions attribute: True
EXT has permissions attribute: True
Experiment 25: All test cases passed.


In [10]:
CLUSTER_SIZE = 4096 # typical disk cluster size in bytes

def calculate_slack_space(file_size, cluster_size=CLUSTER_SIZE):
    clusters_needed = -(-file_size // cluster_size) # ceiling division
    allocated_space = clusters_needed * cluster_size
    slack_space = allocated_space - file_size
    return allocated_space, slack_space

file_size = 5000 # bytes
allocated, slack = calculate_slack_space(file_size)
print(f"File size: {file_size} bytes -> allocated: {allocated} bytes, slack space: {slack} bytes")

def test_experiment26():
    assert allocated == 8192, "5000 bytes needs 2 clusters of 4096 bytes = 8192 bytes allocated"
    assert slack == 8192 - 5000, "Slack space should be the unused remainder of the last cluster"
    allocated2, slack2 = calculate_slack_space(4096)
    assert slack2 == 0, "A file that exactly fills a cluster should have zero slack space"
    print("Experiment 26: All test cases passed.")

test_experiment26()


File size: 5000 bytes -> allocated: 8192 bytes, slack space: 3192 bytes
Experiment 26: All test cases passed.


In [11]:
import hashlib
import json

def acquire(path):
    with open(path, "rb") as f:
        return f.read()

def hash_data(data):
    return hashlib.sha256(data).hexdigest()

def verify_integrity(data, expected_hash):
    return hash_data(data) == expected_hash

def analyze(data, keyword: bytes):
    return keyword in data

def generate_report(case_id, file_hash, keyword_found):
    return {
        "case_id": case_id,
        "sha256": file_hash,
        "suspicious_keyword_found": keyword_found,
        "status": "Evidence Verified" if keyword_found else "No Match",
    }

evidence_path = "workflow_evidence.txt"
with open(evidence_path, "w") as f:
    f.write("Transfer $50000 to account 99881122 immediately, do not report.")

acquired_data = acquire(evidence_path)
evidence_hash = hash_data(acquired_data)
integrity_ok = verify_integrity(acquired_data, evidence_hash)
keyword_found = analyze(acquired_data, b"99881122")
report = generate_report("CASE-2026-014", evidence_hash, keyword_found)

print(json.dumps(report, indent=2))

def test_experiment27():
    assert integrity_ok is True, "Hash verification must succeed on unaltered evidence"
    assert keyword_found is True, "Analysis must detect the suspicious account number"
    assert report["status"] == "Evidence Verified"
    tampered = acquired_data.replace(b"50000", b"99999")
    assert verify_integrity(tampered, evidence_hash) is False, "Tampered evidence must fail hash verification"
    print("Experiment 27: All test cases passed.")

test_experiment27()


{
  "case_id": "CASE-2026-014",
  "sha256": "5761dda7d7476e1b92c42bcb9454f9c018f7b70adcac81879551e70c6f42f4a1",
  "suspicious_keyword_found": true,
  "status": "Evidence Verified"
}
Experiment 27: All test cases passed.


In [12]:
import subprocess
import sys
import os

scripts = [
    "Exp-17-Digital-Evidence-Collection.py",
    "Exp-18-Volatile-vs-Non-Volatile.py",
    "Exp-19-Physical-Logical-Sparse-Acquisition.py",
    "Exp-20-Live-vs-Dead-Acquisition.py",
    "Exp-21-Imaging-vs-Duplication.py",
    "Exp-22-Bit-Stream-Copy-Verification.py",
    "Exp-23-Hashing-and-Avalanche.py",
    "Exp-24-File-Carving.py",
    "Exp-25-File-System-Feature-Simulation.py",
    "Exp-26-Slack-Space-Calculation.py",
    "Exp-27-End-to-End-Forensic-Workflow.py"
]

def run_all():
    print("Running all experiments 17 through 27...\n")
    all_passed = True
    for script in scripts:
        print(f"=== Running {script} ===")
        if not os.path.exists(script):
            print(f"Error: {script} not found in the current directory.")
            all_passed = False
            continue

        result = subprocess.run([sys.executable, script], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Error executing {script}:\n{result.stderr}")
            all_passed = False
        else:
            print(result.stdout)

    if all_passed:
        print("\nAll 11 experiments passed their test cases successfully.")
    else:
        print("\nSome experiments failed.")

if __name__ == "__main__":
    run_all()


Running all experiments 17 through 27...

=== Running Exp-17-Digital-Evidence-Collection.py ===
Error: Exp-17-Digital-Evidence-Collection.py not found in the current directory.
=== Running Exp-18-Volatile-vs-Non-Volatile.py ===
Error: Exp-18-Volatile-vs-Non-Volatile.py not found in the current directory.
=== Running Exp-19-Physical-Logical-Sparse-Acquisition.py ===
Error: Exp-19-Physical-Logical-Sparse-Acquisition.py not found in the current directory.
=== Running Exp-20-Live-vs-Dead-Acquisition.py ===
Error: Exp-20-Live-vs-Dead-Acquisition.py not found in the current directory.
=== Running Exp-21-Imaging-vs-Duplication.py ===
Error: Exp-21-Imaging-vs-Duplication.py not found in the current directory.
=== Running Exp-22-Bit-Stream-Copy-Verification.py ===
Error: Exp-22-Bit-Stream-Copy-Verification.py not found in the current directory.
=== Running Exp-23-Hashing-and-Avalanche.py ===
Error: Exp-23-Hashing-and-Avalanche.py not found in the current directory.
=== Running Exp-24-File-Carvi